# Analyze trained models again
`settings_analysis.py` decides what is computed from the models trained with `settings_training.py`. Change it, for example add metrics or plots, and analyze again. Analysis never trains: the saved models are reused, and so is cached analysis data. Train the profile first (`01_run.ipynb`, or `python run.py train --profile smoke`).

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "settings_training.py").is_file() and (p / "nnpd").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nnpd import execute, load_settings, restore
from nnpd.results import runs

PROFILE = "smoke"  # or "default" / "paper"; use the same profile in all three notebooks
OUTPUT = load_settings("settings_training.py", PROFILE)["output"]

In [ ]:
paths = execute("analyze", profile=PROFILE)  # same as: python run.py analyze --profile smoke
[(record["member"]["prior"], sorted(record["metrics"])) for record in runs(OUTPUT)]

## Try new metrics before adding them
`gaussian/extensions.py` defines two new metrics and a plot. They can be computed directly on a saved run, without changing any file or saving anything:

In [ ]:
from gaussian.extensions import ExtendedAnalysis
context = restore(paths[0])
extension = ExtendedAnalysis()
hook = extension.metrics()["median_absolute_bias"]
hook.compute(context, context.dependencies(hook.needs))

In [ ]:
hook = extension.metrics()["parameter_count"]
hook.compute(context, context.dependencies(hook.needs))

To compute them in every analysis and save them with the results, change `settings_analysis.py`:

```python
"analysis": "gaussian.extensions:ExtendedAnalysis",
"metrics": [..., "median_absolute_bias", "parameter_count"],
"plots": [..., "observation_histogram"],
```

and run the first code cell of this notebook again. For a new shared dataset, register a `Product(builder, needs=(...), settings=...)` in your analysis class. See `docs/EXTENDING.md` for the complete interface and cache-key contract.